# E011 (fast, ~2 h, CPU, no GPU): twin-business stage 2 on E009's stage 1 + learned blocking

**Builds on E009's saved stage 1** (`oof.parquet`: out-of-fold p1 of the 14.5M sampled train pairs;
`test_pairs_proba.parquet`: p1 of all 114M test pairs). No candidate table, no blocking, no stage-1 features:
only stage 2 is trained (`--no-ctx`). Includes E010's stage-2 group consensus and name-ambiguity counts.
**This notebook deletes nothing.**

Finding (E009 OOF errors joined to the train ground truth): most false merges are synthetic **twin businesses**
owned by no S1. A twin copies a real business and changes its **legal form** (record adds a legal form the S1
lacks: twins 45.9% vs true pairs 6.6%; India 39.8% vs 3.2%), adds or swaps a **business word** (52% vs 25%) and
nudges the **house number's last digits**. Our pair features could not see this: `name_core` drops legal forms
and token_set_ratio scores "X" vs "X LLC" as 100. Generator noise words (Center, Services, Shri, Mr, www) point
the other way (true 17% vs twins 9%).

| stage-2 addition | |
|---|---|
| legal forms | canonical codes from the name (Private/Pvt/praivet, Limited, Public, LLC, LLP, Inc, Corp, Co, LP, PC, SARL, SAS, SASU, EURL, SA, SCI, SNC, EI, Ets, ...): added / dropped / same / Private<->Public flip |
| business words | skeleton-word signature without legal forms and noise words: words added / missing / same; noise words added |
| house number | equal, magnitude of the difference, which digit changed (typos hit any digit, twins the last ones) |
| name + legal ambiguity | # S1 carrying exactly this core name AND legal forms |
| groups | the S1's candidates grouped by legal forms and by full profile (house number + legal + words) |
| learned blocking | stage 1 as a filter: drop pairs below the p1 quantile that loses `PRUNE_LOSS` of the true candidate pairs, never below candidate recall `MIN_CAND_RECALL` = 0.98. `candidate_pairs.tsv` = exactly the pairs stage 2 scores |

Run in the **same Kaggle notebook** (needs `work/prepared/*` and `work/experiments/20260926-E009-v4/`).
Accelerator None. Persistence Files. *Run all* resumes.

In [ ]:
# 1. Config
EXP       = "20260927-E011-profile"
SRC_EXP   = "20260926-E009-v4"      # stage-1 source (its oof.parquet + test_pairs_proba.parquet)
PRUNE_LOSS      = 0.001             # share of true candidate pairs learned blocking may drop
MIN_CAND_RECALL = 0.98              # hard floor for candidate recall after pruning
E007_SPEC = "base,conj:20,bm25:5,bge_native:5,name_noaddr:5"
CV_FRAC   = 0.1
LR        = 0.1
REPO      = "https://github.com/Bexwane/AmazonMLchallenge.git"
CODE_DIR  = "/kaggle/working/ber"
WORK      = "/kaggle/working/work"

In [ ]:
# 2. Dataset, validator
import glob, os
hits = glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True)
assert hits, "Dataset not found"
DATA = os.path.dirname(os.path.dirname(hits[0]))
val = glob.glob("/kaggle/input/**/validate_submission.py", recursive=True)
VALIDATOR = val[0] if val else None
for f in ("oof.parquet", "test_pairs_proba.parquet", "stack_metrics.json"):
    assert os.path.exists(f"{WORK}/experiments/{SRC_EXP}/{f}"), f"stage-1 source missing: {f}"
for split in ("train", "test"):
    for f in ("s1.parquet", "s23.parquet"):
        assert os.path.exists(f"{WORK}/prepared/{split}/{f}"), f"prepared data missing: {split}/{f}"
assert os.path.exists(f"{WORK}/prepared/test/NORM_V2"), "test not re-prepared with the France rules"
print("DATA =", DATA, "| VALIDATOR =", VALIDATOR)
!du -sh /kaggle/working; free -g; nproc

In [ ]:
# 3. Code, dependencies, tests
!rm -rf {CODE_DIR} && git clone -q {REPO} {CODE_DIR} && cd {CODE_DIR} && git log --oneline -1
!pip install -q rapidfuzz==3.14.6
import sys; sys.path.insert(0, f"{CODE_DIR}/src")
!cd {CODE_DIR} && python -m pytest -q tests

In [ ]:
# helper
import subprocess, time, json
import pandas as pd
def ber(cmd, *extra):
    args = ["python", "-m", "ber.run", cmd, "--data", DATA, "--work", WORK, "--exp", EXP, "--cv-frac", str(CV_FRAC),
            "--lr", str(LR), "--df-cap", "2500", "--channels", E007_SPEC, "--feat", "v5", "--dense-model", "none",
            "--p1-from", SRC_EXP, "--no-ctx", "--stack-feat", "v2", *map(str, extra)]
    t = time.time()
    p = subprocess.Popen(args, cwd=CODE_DIR, env={**os.environ, "PYTHONPATH": "src"},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0, f"{cmd} failed (exit {p.returncode}); -9 means out of memory"
    print(f"--- {cmd} done in {(time.time() - t) / 60:.1f} min")
X = lambda name: f"{WORK}/experiments/{EXP}/{name}"
cols = ["macro_f05", "micro_precision", "micro_recall", "f05_singletons", "f05_nonsingletons"]

In [ ]:
# 4. Stage 2 with profiles on E009's out-of-fold p1 (+ learned blocking) and selection study
if not os.path.exists(X("stack_metrics.json")):
    ber("stack", "--folds", 3, "--prune-loss", PRUNE_LOSS, "--min-cand-recall", MIN_CAND_RECALL)
sm = json.load(open(X("stack_metrics.json")))
s10 = json.load(open(f"{WORK}/experiments/{SRC_EXP}/stack_metrics.json"))
print("learned blocking:", json.dumps(sm["prune"], indent=1))
assert sm["prune"] is None or sm["prune"]["cand_recall"] >= MIN_CAND_RECALL
rows = [("E009 chosen (LB C 0.950)", s10["chosen"]["macro_f05"]), ("E011 stage 1 (E009 p1, pruned) best selection", sm["stage1"]["macro_f05"]),
        ("E011 stage 2 best selection", sm["stage2"]["macro_f05"]), ("E011 chosen", sm["chosen"]["macro_f05"])]
display(pd.DataFrame(rows, columns=["variant", "CV macro F0.5"]).round(5))
print("use_stack:", sm["use_stack"], "| rule:", {k: v for k, v in sm["rule"].items() if k != "cal"})
display(pd.concat({"E011": pd.DataFrame({k: sm[k] for k in sm if k.startswith(("chosen", "slice_"))}).T[cols + ["n_s1"]],
                   "E009": pd.DataFrame({k: s10[k] for k in s10 if k.startswith(("chosen", "slice_"))}).T[cols]}, axis=1).round(4))
print("new features", {k: v for k, v in sm["stack_feature_gain_top"].items() if k.startswith(("lg_", "w_", "nz_", "hn_", "namelg", "grp_", "l_name", "r_name"))})
print("top", list(sm["stack_feature_gain_top"].items())[:15])

In [ ]:
# 5. Test: stage 2 on E009's p1 (pruned) and three submissions (C = count matching in every country, E009's best)
OUTS = {"A": ("none", "/kaggle/working/output_E011_A"), "B": ("unseen", "/kaggle/working/output_E011_B"),
        "C": ("all", "/kaggle/working/output_E011_C")}
if not os.path.exists(X("test_pairs_proba.parquet")):
    ber("predict", "--out", OUTS["A"][1])
for k, (cm, od) in OUTS.items():
    if not os.path.exists(f"{od}/matching_results.tsv"):
        ber("select", "--count-match", cm, "--out", od, *(["--cand-from", OUTS["A"][1]] if k != "A" else []))
    if VALIDATOR:
        !python {VALIDATOR} --matching {od}/matching_results.tsv --candidate {od}/candidate_pairs.tsv --test-dir {DATA}/test --check-ids | tail -2
    info = json.load(open(f"{od}/selection_info.json"))
    print(k, "count_match =", cm, "| delta", info["delta"])
    display(pd.DataFrame(info["per_country"]).T.round(3))
T = pd.read_parquet(X("test_pairs_proba.parquet"), columns=["s1_idx"])
print("test candidate pairs after learned blocking:", len(T), f"({len(T) / 1732544:.1f} per S1)")

In [ ]:
# 6. Bundle: send /kaggle/working/E011_results.tgz back
import shutil, tarfile
B = "/kaggle/working/E011_results"
os.makedirs(B, exist_ok=True)
for f in ("stack_metrics.json", "oof_errors.parquet"):
    if os.path.exists(X(f)):
        shutil.copy(X(f), f"{B}/{f}")

for k, (_, od) in OUTS.items():
    if os.path.exists(f"{od}/selection_info.json"):
        shutil.copy(f"{od}/selection_info.json", f"{B}/selection_info_{k}.json")
with tarfile.open("/kaggle/working/E011_results.tgz", "w:gz") as t:
    t.add(B, arcname="E011_results")
print(sorted(os.listdir(B)), os.path.getsize("/kaggle/working/E011_results.tgz") // 1024, "KiB")